In [1]:
from utils import create_new_dataset_with_ipcw_weights, create_surv_data, train_test_split_into_df
from lifelines import KaplanMeierFitter, WeibullAFTFitter
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### create dataset

In [2]:
n = 1000
seed = 42

data = create_surv_data(shape_weibull=1,   # constant hazard
                        scale_weibull_base=10000, 
                        rate_censoring=0.01,  
                        b_bloodp=-0.405, 
                        b_diab=-0.4, 
                        b_age=-0.05, 
                        b_bmi=-0.01, 
                        b_kreat=-0.2,
                        n=n, 
                        seed=seed)

Data shape: (1000, 7)
34.0 % of the data has an event


### stratified train and test split

In [3]:
X = data[['bmi', 'blood_pressure', 'kreatinkinase', 'diabetes', 'age']]
y = Surv.from_arrays(event=data['event'], time=data['t'])

df_train, df_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)
df_train, df_test = train_test_split_into_df(df_train=df_train, df_test=df_test, y_train=y_train, y_test=y_test)

In [4]:
# fig, axes = plt.subplots(1, 2, figsize=(6, 4), sharey=True)
# sns.boxplot(ax=axes[0], x='event', y='time', data=df_train)
# axes[0].set_title('Training Data - Time by Event')
# axes[0].set_xlabel('Event')
# axes[0].set_ylabel('Time')
# sns.boxplot(ax=axes[1], x='event', y='time', data=df_test)
# axes[1].set_title('Test Data - Time by Event')
# axes[1].set_xlabel('Event')
# axes[1].set_ylabel('Time')
# y_min = min(df_train['time'].min(), df_test['time'].min())
# y_max = max(df_train['time'].max(), df_test['time'].max())
# axes[0].set_ylim(y_min, y_max)
# axes[1].set_ylim(y_min, y_max)
# plt.tight_layout()
# plt.show()

# train_event_counts = df_train['event'].value_counts(normalize=True)
# test_event_counts = df_test['event'].value_counts(normalize=True)
# print("Relative Häufigkeiten der Events - Trainingsdaten:")
# print(train_event_counts)
# print("\nRelative Häufigkeiten der Events - Testdaten:")
# print(test_event_counts)

# strg k str u  -  kommentar
# strg k str c  -  kommentar entfernen

### cut data at tau // ipcw weights


In [6]:
tau = 12
df_train = create_new_dataset_with_ipcw_weights(df_train,tau)

# portion of censored data after cutpoint
portions_at_cutpoint = df_train['survived'].value_counts(normalize=True)
censored_at_t = portions_at_cutpoint[999]
print(f'censored after cut at tau={tau}: {censored_at_t*100:.2f} %')

print('\n')
print(df_train[df_train['survived']==0].sample(2)[['time', 'event','weights_ipcw','survived', ]])
print('\n')
print(df_train[df_train['survived']==1].sample(2)[['time', 'event','weights_ipcw','survived', ]])
print('\n')
print(df_train[df_train['survived']==999].sample(2)[['time', 'event','weights_ipcw','survived', ]])
print('\n')
print(df_train.columns)


censored after cut at tau=12: 12.43 %


          time  event  weights_ipcw  survived
395   4.126467   True      0.001498         0
22   10.632934   True      0.001625         0


          time  event  weights_ipcw  survived
253  45.879644   True      0.001639         1
518  24.252935  False      0.001639         1


         time  event  weights_ipcw  survived
697  7.640962  False           0.0       999
154  3.489988  False           0.0       999


Index(['bmi', 'blood_pressure', 'kreatinkinase', 'diabetes', 'age', 'event',
       'time', 'survived', 'weights_ipcw'],
      dtype='object')


### Random Forest Modell

In [ ]:
# Random Foreest für Binäre Klassifikation
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer
from sksurv.metrics import concordance_index_censored
from sksurv.metrics import concordance_index_ipcw
from sksurv.metrics import brier_score
from sksurv.metrics import cumulative_dynamic_auc

# Random Forest Classifier
rf = RandomForestClassifier(random_state=seed)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
scoring = {
    'C-index': make_scorer(concordance_index_censored, event_times=[tau]),
    'C-index IPCW': make_scorer(concordance_index_ipcw, tau=tau),
    'Brier Score': make_scorer(brier_score),
    'CDA': make_scorer(cumulative_dynamic_auc, tau=tau)
}
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring=scoring, refit='C-index IPCW')

grid_search.fit(df_train.drop(['time', 'event', 'weights_ipcw', 'survived'], axis=1), df_train[['event', 'time']])


### Weibull Modell fitten

In [387]:
aft = WeibullAFTFitter()
aft.fit(df_train, duration_col='time', event_col='event')
aft.print_summary()

""" df = create_surv_data(shape_weibull=1,   # constant hazard
                        scale_weibull_base=10000, 
                        rate_censoring=0.01,  
                        b_bloodp=-0.405, 
                        b_diab=-0.4, 
                        b_age=-0.05, 
                        b_bmi=-0.01, 
                        b_kreat=-0.2,
                        n=n, 
                        seed=42) """

#aft.plot()

<lifelines.WeibullAFTFitter: fitted with 700 total observations, 462 right-censored observations>
             duration col = 'time'
                event col = 'event'
   number of observations = 700
number of events observed = 238
           log-likelihood = -1426.19
         time fit was run = 2024-07-29 20:13:51 UTC

---
                        coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
param   covariate                                                                                                       
lambda_ age            -0.05      0.95      0.01           -0.07           -0.04                0.94                0.96
        blood_pressure -0.43      0.65      0.14           -0.70           -0.17                0.50                0.84
        bmi            -0.01      0.99      0.00           -0.01           -0.01                0.99                0.99
        diabetes       -0.64      0.53      0.15           -0.94           -0.34                0.39                0.71
        kreatinkinase  -0.23      0.80      0.07           -0.36           -0.09                0.70                0.91
        Intercept       9.53  13701.26      0.55            8.44           10.61             4645.77            40407.57
rho_    Intercept       0.00      1.00      0.05           -0.10            0.10                0.91                1.10

                        cmp to     z      p  -log2(p)
param   covariate                                    
lambda_ age               0.00 -7.10 <0.005     39.55
        blood_pressure    0.00 -3.21 <0.005      9.57
        bmi               0.00 -5.69 <0.005     26.22
        diabetes          0.00 -4.23 <0.005     15.40
        kreatinkinase     0.00 -3.33 <0.005     10.18
        Intercept         0.00 17.26 <0.005    219.38
rho_    Intercept         0.00  0.01   0.99      0.01
---
Concordance = 0.69
AIC = 2866.38
log-likelihood ratio test = 113.36 on 5 df
-log2(p) of ll-ratio test = 73.41

' df = create_surv_data(shape_weibull=1,   # constant hazard\n                        scale_weibull_base=10000, \n                        rate_censoring=0.01,  \n                        b_bloodp=-0.405, \n                        b_diab=-0.4, \n                        b_age=-0.05, \n                        b_bmi=-0.01, \n                        b_kreat=-0.2,\n                        n=n, \n                        seed=42) '

### Weibull Modell evaluation

In [388]:
tied_risk_scores = aft.predict_expectation(df_train)

# IPCW-C-Index berechnen
ci_ipcw, concordant, discordant, tied_risk, tied_time = concordance_index_ipcw(
    y_train, y_train, -tied_risk_scores )


print("IPCW-C-Index:", ci_ipcw)


IPCW-C-Index: 0.6740393506695707


In [389]:
# Kaplan-Meier-Schätzung für Zensierverteilung
kmf = KaplanMeierFitter()
kmf.fit(df['t'], event_observed=1-df['event'])
censoring_prob = 1 - kmf.survival_function_at_times([t]).values[0]
censoring_prob_after_t = 1-censoring_prob
print(censoring_prob_after_t)

# IPC-Gewichte 



df['ipc_weight'] = 1 / censoring_prob_after_t
#data_bi.loc[data['event'] == 0, 'ipc_weight'] = 


# cutpoint for the time t
t = 150
data_bi = create_new_dataset(df,t)
data_bi['survived'].value_counts(normalize=True)

NameError: name 'data' is not defined

In [14]:
data_bi['ipc_weight'].sum()

1.5380413332677234